# Web Scraping: Largest Companies in the United States

This project demonstrates how to collect structured data from a web page using Python, BeautifulSoup, Requests, and pandas.

The source is the Wikipedia page **List of largest companies in the United States by revenue**. The project extracts the table containing the 100 largest companies and stores it in a pandas DataFrame for further analysis.

## 1. Import libraries

The project uses:
- `requests` to download the web page;
- `BeautifulSoup` to parse the HTML and locate the table;
- `pandas` to store the scraped data in a structured DataFrame.

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

## 2. Define the source URL

The data is collected directly from the Wikipedia page containing the ranking.

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_largest_companies_in_the_United_States_by_revenue'

## 3. Send an HTTP request

A browser-like `User-Agent` is included so the request resembles a normal browser request. `timeout` prevents the request from waiting indefinitely, and `raise_for_status()` makes HTTP errors visible instead of silently continuing.

In [3]:
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_5) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/50.0.2661.102 Safari/537.36'
    )
}

page = requests.get(url, headers=headers, timeout=30)
page.raise_for_status()
page

<Response [200]>

## 4. Parse the HTML page

BeautifulSoup converts the downloaded HTML into a structure that can be searched. Here we use Python's built-in `html.parser`.

In [4]:
soup = BeautifulSoup(page.text, 'html.parser')

## 5. Locate the table

The first HTML table on the page is selected for scraping.

In [5]:
table = soup.find('table')

if table is None:
    raise ValueError('Could not find a table on the source page.')

## 6. Extract column names

The table headers are collected from the `<th>` HTML elements and will be used as DataFrame column names.

In [6]:
table_headers = [x.text.strip() for x in table.find_all('th')]
table_headers

['Rank',
 'Name',
 'Industry',
 'Revenue (USD millions)',
 'Revenue growth',
 'Employees',
 'Headquarters']

## 7. Create an empty DataFrame

The DataFrame is initialized with the column names extracted from the HTML table.

In [7]:
df = pd.DataFrame(columns=table_headers)
df

,Rank,Name,Industry,Revenue (USD millions),Revenue growth,Employees,Headquarters


## 8. Inspect the first data row

Before scraping all rows, the first row is inspected to understand the HTML structure and confirm that the expected cells are present.

In [8]:
rows = table.find_all('tr')
first_row = rows[1].find_all('td')
first_row

[<td id="mwSA">1</td>,
 <td id="mwSQ"><a href="https://en.wikipedia.org/wiki/Walmart" id="mwSg" rel="mw:WikiLink" title="Walmart">Walmart</a></td>,
 <td id="mwSw">Retail</td>,
 <td id="mwTA" style="text-align:center;">680,985</td>,
 <td id="mwTQ" style="text-align:center;"><span about="#mwt7" data-mw='{"caption":"Increase","parts":[{"template":{"target":{"wt":"profit","href":"./Template:Profit"},"params":{},"i":0}}]}' id="mwTg" typeof="mw:Transclusion mw:File"><span title="Increase"><img alt="Increase" class="mw-file-element" data-file-height="300" data-file-type="drawing" data-file-width="300" decoding="async" height="11" loading="lazy" resource="https://en.wikipedia.org/wiki/File:Increase2.svg" src="//thumb.wikimedia.org/wikipedia/commons/thumb/b/b0/Increase2.svg/20px-Increase2.svg.png?utm_source=en.wikipedia.org&amp;utm_campaign=parser&amp;utm_content=thumbnail" srcset="//thumb.wikimedia.org/wikipedia/commons/thumb/b/b0/Increase2.svg/40px-Increase2.svg.png?utm_source=en.wikipedia.or

## 9. Extract all table rows

Each table row is converted into a list of cell values. Rows with a different number of cells are skipped so that the resulting DataFrame keeps a consistent structure.

In [9]:
for row in rows[1:]:
    table_row = [data.get_text(strip=True) for data in row.find_all('td')]

    # Keep only rows that match the number of expected columns.
    if len(table_row) == len(table_headers):
        df.loc[len(df)] = table_row

df

,Rank,Name,Industry,Revenue (USD millions),Revenue growth,Employees,Headquarters
0,1,Walmart,Retail,"680,985",5.1%,"2,100,000","Bentonville, Arkansas"
1,2,Amazon,Retail and cloud computing,"637,959",11.0%,"1,556,000","Seattle, Washington"
2,3,UnitedHealth Group,Healthcare,"400,278",7.7%,"400,000","Minnetonka, Minnesota"
3,4,Apple,Technology,"391,035",2.0%,"164,000","Cupertino, California"
4,5,CVS Health,Healthcare,"372,809",4.2%,"259,500","Woonsocket, Rhode Island"
...,...,...,...,...,...,...,...
95,96,General Dynamics,Aerospace and defense,"47,716",12.9%,"117,000","Reston, Virginia"
96,97,Coca-Cola,Beverage,"47,061",2.9%,"69,700","Atlanta, Georgia"
97,98,TIAA,Financials,"46,946",2.6%,"15,623","New York City, New York"
98,99,Travelers,Insurance,"46,423",12.2%,"34,000","New York City, New York"


## Result

The final DataFrame contains the scraped company ranking with the following fields:

- Rank
- Name
- Industry
- Revenue (USD millions)
- Revenue growth
- Employees
- Headquarters


In [10]:
df.shape


(100, 7)